# multimodal_encoder — Colab Notebook

End-to-end fused transformer: candle stream + indicator stream concatenated
per time-step, projected to a shared hidden dim, run through a 4-layer
transformer with CLS pooling, then classified into 3 classes.

Self-contained. Recommended runtime: **T4 GPU**. Run all cells top-to-bottom.


In [ ]:
!pip install -q pyarrow polars pyyaml scikit-learn pandas
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


## 1. Tokenizers (delta + bucket)


In [ ]:
"""Shared tokenizers (copied from token_first_transformer/tokenizer/)."""
from __future__ import annotations

from pathlib import Path

import numpy as np


class DeltaTokenizer:
    """Percentage-delta tokenizer (0=PAD, 1=CLS, 2+=bins)."""

    def __init__(self, range_pct: float = 3.0, step_pct: float = 0.05) -> None:
        self.range_pct = range_pct
        self.step_pct = step_pct
        self.n_bins = int(2 * range_pct / step_pct)
        self.pad_id = 0
        self.cls_id = 1
        self._offset = 2
        self.vocab_size = self._offset + self.n_bins

    def encode_batch(self, deltas: np.ndarray) -> np.ndarray:
        deltas_pct = deltas * 100
        bin_idx = np.round(deltas_pct / self.step_pct).astype(np.int32) + self.n_bins // 2 - 1
        bin_idx = np.clip(bin_idx, 0, self.n_bins - 1)
        return (bin_idx + self._offset).astype(np.int32)

    def from_closes(self, closes: np.ndarray) -> np.ndarray:
        n = len(closes)
        ids = np.full(n, self.pad_id, dtype=np.int32)
        if n < 2:
            return ids
        deltas = (closes[1:] - closes[:-1]) / closes[:-1]
        ids[1:] = self.encode_batch(deltas)
        return ids


class BucketTokenizer:
    """Quantile bucket tokenizer (0=PAD, 1=reserved CLS, 2+=bins)."""

    def __init__(self, n_bins: int = 8) -> None:
        self.n_bins = n_bins
        self.pad_id = 0
        self._offset = 2
        self.vocab_size = self._offset + n_bins
        self.boundaries = None

    def fit(self, values: np.ndarray) -> None:
        quantiles = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        self.boundaries = np.percentile(values, quantiles).astype(np.float32)

    def encode_batch(self, values: np.ndarray) -> np.ndarray:
        assert self.boundaries is not None
        bin_idx = np.searchsorted(self.boundaries, values, side="right")
        return (bin_idx + self._offset).astype(np.int32)


## 2. Indicator computer


In [ ]:
"""Technical-indicator computer (from indicator_tokenizer/indicators/computer.py)."""
import numpy as np


def _ema(arr, span):
    alpha = 2.0 / (span + 1)
    out = np.zeros(len(arr), dtype=np.float64)
    out[0] = arr[0]
    for i in range(1, len(arr)):
        out[i] = alpha * arr[i] + (1 - alpha) * out[i - 1]
    return out


class IndicatorComputer:
    def rsi(self, close, period=14):
        delta = np.diff(close.astype(np.float64), prepend=close[0])
        gain = np.where(delta > 0, delta, 0.0)
        loss = np.where(delta < 0, -delta, 0.0)
        avg_gain = _ema(gain, period)
        avg_loss = _ema(loss, period)
        rs = avg_gain / (avg_loss + 1e-10)
        return (100 - 100 / (1 + rs)).astype(np.float32)

    def macd_hist(self, close, fast=12, slow=26, signal=9):
        c = close.astype(np.float64)
        macd_line = _ema(c, fast) - _ema(c, slow)
        signal_line = _ema(macd_line, signal)
        return (macd_line - signal_line).astype(np.float32)

    def bollinger_pctb(self, close, period=20, num_std=2.0):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            w = c[i - period + 1 : i + 1]
            sma = w.mean(); std = w.std()
            upper = sma + num_std * std; lower = sma - num_std * std
            out[i] = (c[i] - lower) / (upper - lower + 1e-10)
        return out

    def atr(self, high, low, close, period=14):
        tr = np.zeros(len(close), dtype=np.float64)
        tr[0] = high[0] - low[0]
        tr[1:] = np.maximum(
            high[1:] - low[1:],
            np.maximum(
                np.abs(high[1:].astype(np.float64) - close[:-1].astype(np.float64)),
                np.abs(low[1:].astype(np.float64) - close[:-1].astype(np.float64)),
            ),
        )
        return _ema(tr, period).astype(np.float32)

    def volume_ratio(self, close, volume, period=20):
        v = volume.astype(np.float64)
        out = np.zeros(len(v), dtype=np.float32)
        for i in range(period - 1, len(v)):
            sma = v[i - period + 1 : i + 1].mean()
            out[i] = v[i] / (sma + 1e-10)
        return out

    def price_vs_sma(self, close, period=20):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            sma = c[i - period + 1 : i + 1].mean()
            out[i] = (c[i] - sma) / (sma + 1e-10)
        return out

    def compute_all(self, ohlcv):
        return {
            "rsi": self.rsi(ohlcv["close"]),
            "macd_hist": self.macd_hist(ohlcv["close"]),
            "bollinger_pctb": self.bollinger_pctb(ohlcv["close"]),
            "atr": self.atr(ohlcv["high"], ohlcv["low"], ohlcv["close"]),
            "volume_ratio": self.volume_ratio(ohlcv["close"], ohlcv["volume"]),
            "price_vs_sma": self.price_vs_sma(ohlcv["close"]),
        }


## 3. Indicator tokenizer


In [ ]:
"""Indicator tokenizer (from indicator_tokenizer/indicators/tokenizer.py)."""
from pathlib import Path
import numpy as np


class FixedBoundaries:
    def __init__(self, bins, offset=2):
        self.bins = np.array(bins, dtype=np.float32)
        self.offset = offset
        self.vocab_size = len(bins) + 1 + offset

    def encode_batch(self, values):
        return (np.searchsorted(self.bins, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.bins)
    def load(self, path): self.bins = np.load(path)


class QuantileBoundaries:
    def __init__(self, n_bins, offset=2):
        self.n_bins = n_bins
        self.offset = offset
        self.vocab_size = n_bins + offset
        self.boundaries = None

    def fit(self, values):
        q = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        self.boundaries = np.percentile(values, q).astype(np.float32)

    def encode_batch(self, values):
        assert self.boundaries is not None
        return (np.searchsorted(self.boundaries, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.boundaries)
    def load(self, path): self.boundaries = np.load(path)


class IndicatorTokenizer:
    PAD_ID = 0; SPECIAL_ID = 1

    def __init__(self):
        self.rsi = FixedBoundaries(bins=[20, 30, 70, 80])
        self.macd_hist = QuantileBoundaries(n_bins=7)
        self.bollinger_pctb = FixedBoundaries(bins=[0.0, 0.25, 0.75, 1.0])
        self.atr = QuantileBoundaries(n_bins=6)
        self.volume_ratio = QuantileBoundaries(n_bins=5)
        self.price_vs_sma = QuantileBoundaries(n_bins=5)
        self._quantile_fields = ["macd_hist", "atr", "volume_ratio", "price_vs_sma"]
        self._all_fields = ["rsi", "macd_hist", "bollinger_pctb", "atr",
                            "volume_ratio", "price_vs_sma"]

    def fit(self, ind):
        for f in self._quantile_fields:
            getattr(self, f).fit(ind[f])

    def encode(self, ind):
        return {f: getattr(self, f).encode_batch(ind[f]) for f in self._all_fields}

    def vocab_sizes(self):
        return {f: getattr(self, f).vocab_size for f in self._all_fields}

    def save(self, d):
        d = Path(d); d.mkdir(parents=True, exist_ok=True)
        for f in self._all_fields:
            getattr(self, f).save(d / f"{f}.npy")

    def load(self, d):
        d = Path(d)
        for f in self._all_fields:
            getattr(self, f).load(d / f"{f}.npy")


## 4. Multimodal model


In [ ]:
"""MultimodalEncoder (from multimodal_encoder/models/multimodal_model.py)."""
import torch
import torch.nn as nn


class MultimodalEncoder(nn.Module):
    def __init__(self, delta_vocab_size=122, bucket_vocab_size=10,
                 delta_emb_dim=64, bucket_emb_dim=16, candle_proj_dim=128,
                 ind_vocab_sizes=None, ind_emb_dim=16, ind_proj_dim=128,
                 hidden_dim=256, num_layers=4, num_heads=8, ffn_dim=1024,
                 dropout=0.1, num_classes=3, seq_len=128):
        super().__init__()
        if ind_vocab_sizes is None:
            ind_vocab_sizes = [7, 9, 7, 8, 7, 7]
        self.delta_emb = nn.Embedding(delta_vocab_size, delta_emb_dim)
        self.vol_emb = nn.Embedding(bucket_vocab_size, bucket_emb_dim)
        self.vb_emb = nn.Embedding(bucket_vocab_size, bucket_emb_dim)
        self.candle_proj = nn.Linear(delta_emb_dim + bucket_emb_dim * 2, candle_proj_dim)
        self.ind_embeddings = nn.ModuleList(
            [nn.Embedding(vs, ind_emb_dim) for vs in ind_vocab_sizes])
        self.ind_proj = nn.Linear(ind_emb_dim * len(ind_vocab_sizes), ind_proj_dim)
        self.fusion_proj = nn.Linear(candle_proj_dim + ind_proj_dim, hidden_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        self.pos_emb = nn.Embedding(seq_len + 1, hidden_dim)
        enc = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout, activation="gelu",
            batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, num_classes))

    def forward(self, delta_ids, vol_ids, vb_ids, ind_inputs):
        B = delta_ids.shape[0]
        d = self.delta_emb(delta_ids); v = self.vol_emb(vol_ids); vb = self.vb_emb(vb_ids)
        candle = self.candle_proj(torch.cat([d, v, vb], dim=-1))
        ind_e = [emb(tok) for emb, tok in zip(self.ind_embeddings, ind_inputs)]
        indicator = self.ind_proj(torch.cat(ind_e, dim=-1))
        fused = self.fusion_proj(torch.cat([candle, indicator], dim=-1))
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, fused], dim=1)
        T = x.shape[1]
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_emb(pos)
        x = self.norm(self.transformer(x))
        return self.head(x[:, 0])


## 5. Dataset + collate


In [ ]:
"""MultimodalDataset + collate."""
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq


def _load(p):
    t = pq.read_table(p)
    return {c: np.array([v.as_py() for v in t.column(c)], dtype=np.float32) for c in t.column_names}


def _fit_all(files, range_pct, step_pct, n_bins):
    dt = DeltaTokenizer(range_pct=range_pct, step_pct=step_pct)
    rp, lv = [], []
    for f in files:
        d = _load(f)
        if len(d["close"]) < 2: continue
        rp.append((d["high"][1:] - d["low"][1:]) / d["close"][1:])
        lv.append(np.log1p(d["volume"][1:]))
    vt = BucketTokenizer(n_bins=n_bins); vt.fit(np.concatenate(rp))
    bt = BucketTokenizer(n_bins=n_bins); bt.fit(np.concatenate(lv))
    comp = IndicatorComputer()
    keys = ["rsi","macd_hist","bollinger_pctb","atr","volume_ratio","price_vs_sma"]
    ai = {k: [] for k in keys}
    for f in files:
        d = _load(f)
        ind = comp.compute_all({k2: d[k2] for k2 in ["open","high","low","close","volume"]})
        for k in keys: ai[k].append(ind[k])
    it = IndicatorTokenizer(); it.fit({k: np.concatenate(v) for k, v in ai.items()})
    return dt, vt, bt, it, comp


class MultimodalDataset:
    def __init__(self, files, seq_len=128, target_horizon=60, target_threshold=0.0015,
                 range_pct=3.0, step_pct=0.05, n_bins=8):
        self.seq_len = seq_len
        self.target_horizon = target_horizon
        self.target_threshold = target_threshold
        self.dt, self.vt, self.bt, self.it, self.comp = _fit_all(files, range_pct, step_pct, n_bins)
        frames = [_load(f) for f in files]
        self.closes = np.concatenate([f["close"] for f in frames]).astype(np.float32)
        self.highs = np.concatenate([f["high"] for f in frames]).astype(np.float32)
        self.lows = np.concatenate([f["low"] for f in frames]).astype(np.float32)
        self.volumes = np.concatenate([f["volume"] for f in frames]).astype(np.float32)
        self.opens = np.concatenate([f.get("open", f["close"]) for f in frames]).astype(np.float32)
        self._len = max(0, len(self.closes) - seq_len - target_horizon)

    def __len__(self): return self._len

    def __getitem__(self, idx):
        s, e = idx, idx + self.seq_len
        c, h, l, v, o = (self.closes[s:e], self.highs[s:e], self.lows[s:e],
                         self.volumes[s:e], self.opens[s:e])
        delta = self.dt.from_closes(c); delta[0] = self.dt.cls_id
        rp = np.zeros(self.seq_len, dtype=np.float32)
        rp[1:] = (h[1:] - l[1:]) / c[1:]
        vol = self.vt.encode_batch(rp); vol[0] = self.vt.pad_id
        vb = self.bt.encode_batch(np.log1p(v)); vb[0] = self.bt.pad_id
        ind = self.it.encode(self.comp.compute_all(
            {"open": o, "high": h, "low": l, "close": c, "volume": v}))
        tc = self.closes[e + self.target_horizon - 1]
        cc = self.closes[e - 1]
        d = (tc - cc) / cc
        label = 2 if d > self.target_threshold else (0 if d < -self.target_threshold else 1)
        return delta, vol, vb, ind, label


def collate(batch):
    import torch
    delta = torch.stack([torch.tensor(b[0]) for b in batch]).long()
    vol = torch.stack([torch.tensor(b[1]) for b in batch]).long()
    vb = torch.stack([torch.tensor(b[2]) for b in batch]).long()
    keys = list(batch[0][3].keys())
    ind = [torch.stack([torch.tensor(b[3][k]) for b in batch]).long() for k in keys]
    y = torch.tensor([b[4] for b in batch]).long()
    return delta, vol, vb, ind, y


## 6. Trainer


In [ ]:
"""Trainer for MultimodalEncoder."""
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score


class Trainer:
    def __init__(self, model, train_loader, val_loader,
                 epochs=10, lr=3e-4, weight_decay=0.01, early_stop_patience=3,
                 device="auto", checkpoint_dir=Path("/content/checkpoints")):
        self.device = self._auto_device() if device == "auto" else device
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs
        self.early_stop_patience = early_stop_patience
        self.ckpt_dir = Path(checkpoint_dir); self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=epochs)

    @staticmethod
    def _auto_device():
        if torch.cuda.is_available(): return "cuda"
        if torch.backends.mps.is_available(): return "mps"
        return "cpu"

    def train(self):
        best_f1, pat = -1.0, 0
        for ep in range(1, self.epochs + 1):
            tl = self._train_ep()
            vl, vf = self._val_ep()
            print(f"Ep {ep}: train={tl:.4f} val={vl:.4f} f1={vf:.4f}")
            if vf > best_f1:
                best_f1, pat = vf, 0
                torch.save({"model": self.model.state_dict()}, self.ckpt_dir / "best.pt")
            else:
                pat += 1
            if pat >= self.early_stop_patience:
                print(f"Early stop ep{ep}"); break
            self.scheduler.step()

    def _train_ep(self):
        self.model.train(mode=True); tl, n = 0.0, 0
        for b in self.train_loader:
            delta, vol, vb, ind, y = b
            dev = self.device
            logits = self.model(delta.to(dev), vol.to(dev), vb.to(dev), [t.to(dev) for t in ind])
            loss = self.criterion(logits, y.to(dev))
            loss.backward(); self.optimizer.step(); self.optimizer.zero_grad()
            tl += loss.item(); n += 1
        return tl / max(n, 1)

    def _val_ep(self):
        self.model.train(mode=False)
        tl, n = 0.0, 0; preds, labels = [], []
        with torch.no_grad():
            for b in self.val_loader:
                delta, vol, vb, ind, y = b
                logits = self.model(delta.to(self.device), vol.to(self.device),
                                    vb.to(self.device), [t.to(self.device) for t in ind])
                tl += self.criterion(logits, y.to(self.device)).item(); n += 1
                preds.extend(logits.argmax(-1).cpu().numpy())
                labels.extend(y.numpy())
        return tl/max(n,1), float(f1_score(labels, preds, average="weighted", zero_division=0))


## 7. Configuration + data paths


In [ ]:
"""Configuration."""
from pathlib import Path

USE_MOCK_DATA = True
if USE_MOCK_DATA:
    DATA_DIR = Path("/content/mock_data")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/trading_data")

MOCK_MONTHS = ["2024-01", "2024-02", "2024-03", "2024-04"]
MOCK_MINUTES_PER_MONTH = 20000
SYMBOL = "BTCUSDT"

CFG = {
    "data": {
        "train_months": ["2024-01", "2024-02"],
        "val_months":   ["2024-03", "2024-03"],
        "test_months":  ["2024-04", "2024-04"],
    },
    "sequence": {"length": 128, "target_horizon": 60, "target_threshold": 0.0015},
    "tokenizer": {"delta": {"range_pct": 3.0, "step_pct": 0.05}, "bucket": {"n_bins": 8}},
    "model": {
        "candle": {"delta_vocab_size": 122, "bucket_vocab_size": 10,
                   "delta_emb_dim": 64, "bucket_emb_dim": 16, "proj_dim": 128},
        "indicator": {"vocab_sizes": [7, 9, 7, 8, 7, 7], "emb_dim": 16, "proj_dim": 128},
        "fusion": {"hidden_dim": 256, "num_layers": 4, "num_heads": 8,
                   "ffn_dim": 1024, "dropout": 0.1, "num_classes": 3},
    },
    "training": {"batch_size": 64, "learning_rate": 3e-4, "weight_decay": 0.01,
                 "epochs": 5, "early_stop_patience": 3, "device": "auto",
                 "checkpoint_dir": "/content/checkpoints"},
}

if Path("/content/drive/MyDrive").exists():
    ARTIFACTS_ROOT = Path("/content/drive/MyDrive/w_training/multimodal_encoder")
else:
    ARTIFACTS_ROOT = Path("/content/artifacts/multimodal_encoder")
print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)


## 8. Mock data generation


In [ ]:
"""Generate synthetic BTCUSDT 1-min OHLCV (geometric Brownian motion).
Skipped automatically when USE_MOCK_DATA = False."""
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq


def _gen_month(n_minutes, start_price, seed):
    rng = np.random.default_rng(seed)
    sigma = 0.0008
    lr = rng.normal(0.0, sigma, size=n_minutes)
    closes = start_price * np.exp(np.cumsum(lr))
    opens = np.concatenate([[start_price], closes[:-1]])
    noise = np.abs(rng.normal(0, sigma, size=n_minutes)) * closes
    highs = np.maximum(opens, closes) + noise
    lows = np.minimum(opens, closes) - noise
    vols = rng.lognormal(3.0, 1.0, size=n_minutes).astype(np.float32)
    return {
        "open": opens.astype(np.float32),
        "high": highs.astype(np.float32),
        "low": lows.astype(np.float32),
        "close": closes.astype(np.float32),
        "volume": vols,
    }


if USE_MOCK_DATA:
    kd = DATA_DIR / "BTCUSDT" / "klines_1m"
    kd.mkdir(parents=True, exist_ok=True)
    start_price = 42000.0
    for i, m in enumerate(MOCK_MONTHS):
        out = kd / f"{m}.parquet"
        if out.exists():
            print("skip", out.name); continue
        d = _gen_month(MOCK_MINUTES_PER_MONTH, start_price, 42 + i)
        start_price = float(d["close"][-1])
        pq.write_table(pa.table(d), out)
        print(f"wrote {out.name}  rows={MOCK_MINUTES_PER_MONTH}")
    print("files:", sorted(p.name for p in kd.glob("*.parquet")))
else:
    print("USE_MOCK_DATA=False — expecting real data at", DATA_DIR)


## 9. make_split helper


In [ ]:
"""Helper: discover parquet files for a month range."""
from pathlib import Path


def make_split(data_dir: Path, symbol: str, start_month: str, end_month: str):
    kd = data_dir / symbol / "klines_1m"
    if not kd.exists():
        raise FileNotFoundError(kd)
    return sorted([f for f in kd.glob("*.parquet") if start_month <= f.stem <= end_month])


## 10. Main


In [ ]:
"""Main: dataset -> model -> trainer -> test eval."""
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

sc = CFG["sequence"]; tc = CFG["tokenizer"]
train_files = make_split(DATA_DIR, SYMBOL, *CFG["data"]["train_months"])
val_files   = make_split(DATA_DIR, SYMBOL, *CFG["data"]["val_months"])
test_files  = make_split(DATA_DIR, SYMBOL, *CFG["data"]["test_months"])
print(f"train={len(train_files)} val={len(val_files)} test={len(test_files)}")

mk = dict(seq_len=sc["length"], target_horizon=sc["target_horizon"],
          target_threshold=sc["target_threshold"],
          range_pct=tc["delta"]["range_pct"], step_pct=tc["delta"]["step_pct"],
          n_bins=tc["bucket"]["n_bins"])

train_ds = MultimodalDataset(train_files, **mk)
val_ds   = MultimodalDataset(val_files, **mk)
test_ds  = MultimodalDataset(test_files, **mk)
print(f"train_ds={len(train_ds)} val_ds={len(val_ds)} test_ds={len(test_ds)}")

bs = CFG["training"]["batch_size"]
tr_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  collate_fn=collate, num_workers=0)
va_dl = DataLoader(val_ds,   batch_size=bs, shuffle=False, collate_fn=collate, num_workers=0)
te_dl = DataLoader(test_ds,  batch_size=bs, shuffle=False, collate_fn=collate, num_workers=0)

mc = CFG["model"]["candle"]; mi = CFG["model"]["indicator"]; mf = CFG["model"]["fusion"]
model = MultimodalEncoder(
    delta_vocab_size=mc["delta_vocab_size"], bucket_vocab_size=mc["bucket_vocab_size"],
    delta_emb_dim=mc["delta_emb_dim"], bucket_emb_dim=mc["bucket_emb_dim"],
    candle_proj_dim=mc["proj_dim"],
    ind_vocab_sizes=mi["vocab_sizes"], ind_emb_dim=mi["emb_dim"], ind_proj_dim=mi["proj_dim"],
    hidden_dim=mf["hidden_dim"], num_layers=mf["num_layers"], num_heads=mf["num_heads"],
    ffn_dim=mf["ffn_dim"], dropout=mf["dropout"], num_classes=mf["num_classes"],
    seq_len=sc["length"])
print(f"params={sum(p.numel() for p in model.parameters()):,}")

t = CFG["training"]
trainer = Trainer(model, tr_dl, va_dl, epochs=t["epochs"], lr=t["learning_rate"],
    weight_decay=t["weight_decay"], early_stop_patience=t["early_stop_patience"],
    device=t["device"], checkpoint_dir=Path(t["checkpoint_dir"]))
trainer.train()

print("\n=== Test evaluation ===")
dev = trainer.device
state = torch.load(Path(t["checkpoint_dir"]) / "best.pt", map_location=dev, weights_only=True)
model.load_state_dict(state["model"]); model.train(mode=False)
all_preds, all_labels = [], []
with torch.no_grad():
    for b in te_dl:
        d, vo, vb, ind, y = b
        logits = model(d.to(dev), vo.to(dev), vb.to(dev), [t_.to(dev) for t_ in ind])
        all_preds.extend(logits.argmax(-1).cpu().numpy().tolist())
        all_labels.extend(y.numpy().tolist())
print(classification_report(all_labels, all_preds,
    target_names=["DOWN","FLAT","UP"], zero_division=0))


## 11. Save artifacts to Google Drive
Persists checkpoint, fitted tokenizers, config and test metrics to `ARTIFACTS_ROOT` (Drive or `/content/artifacts`).


In [ ]:
"""Save checkpoint, tokenizers, metrics and config to ARTIFACTS_ROOT."""
import json
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ARTIFACTS_ROOT / f"run_{RUN_TAG}"
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "tokenizers").mkdir(parents=True, exist_ok=True)

src = Path(CFG["training"]["checkpoint_dir"]) / "best.pt"
if src.exists():
    shutil.copy(src, RUN_DIR / "checkpoints" / "best.pt")

np.save(RUN_DIR / "tokenizers" / "vol_boundaries.npy", train_ds.vt.boundaries)
np.save(RUN_DIR / "tokenizers" / "vb_boundaries.npy",  train_ds.bt.boundaries)
with open(RUN_DIR / "tokenizers" / "delta_params.json", "w") as f:
    json.dump({
        "range_pct": train_ds.dt.range_pct,
        "step_pct":  train_ds.dt.step_pct,
    }, f, indent=2)
train_ds.it.save(RUN_DIR / "tokenizers" / "indicators")

with open(RUN_DIR / "config.json", "w") as f:
    json.dump(CFG, f, indent=2, default=str)

test_report_dict = classification_report(
    all_labels, all_preds, target_names=["DOWN","FLAT","UP"],
    zero_division=0, output_dict=True)
with open(RUN_DIR / "test_metrics.json", "w") as f:
    json.dump({
        "report":           test_report_dict,
        "confusion_matrix": confusion_matrix(all_labels, all_preds).tolist(),
    }, f, indent=2)

np.savez_compressed(
    RUN_DIR / "predictions.npz",
    preds=np.asarray(all_preds, dtype=np.int8),
    labels=np.asarray(all_labels, dtype=np.int8),
)

print(f"saved to: {RUN_DIR}")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR)!s:40s}  {p.stat().st_size:>10,} B")
